# URL Shortener

> **Time-box:** 30–45 minutes for schema + API surface.

## Core requirements

1. A client submits a long URL and receives a short code (e.g. `aB3xQ`).
2. `GET /<code>` redirects to the original URL.
3. A client can fetch basic click analytics for a code (total clicks at a minimum).
4. The same long URL submitted twice does **not** have to produce the same code — your call, defend your choice.

## Stretch goals

- Users can sign up; links can be owned by a user.
- A user can request a **custom alias** (`/my-talk-2025`). What happens on collision?
- Links can have an **expiry**.
- Click analytics broken down by day and/or referrer.
- Rate limit link creation per IP / per user.

## Things the interviewer will probe

- **Code generation:** counter + base62? random? hash of URL? trade-offs?
- **Read/write ratio:** redirects dwarf creates by orders of magnitude. What does that imply about indexes? Caching?
- **Hot keys:** one viral link could get most of your traffic. Implications for analytics writes?
- **Status codes:** 201 for create. 301 vs 302 for the redirect — does it matter for analytics?
- **Idempotency:** is `POST /links` idempotent? Should it be?

---
## Setup

In [ ]:
import json
import sqlite3

import pandas as pd
from fastapi import FastAPI
from fastapi.testclient import TestClient
from IPython.display import display
from pydantic import BaseModel

## Schema

Edit the SQL and re-run this cell to get a fresh in-memory database.

In [ ]:
SCHEMA = """
-- Design your tables here
"""

conn = sqlite3.connect(":memory:", check_same_thread=False)
conn.row_factory = sqlite3.Row
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript(SCHEMA)

print("Tables:", [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
).fetchall()])

## API

> After editing any cell below, re-run from **App** down through **Client**.

In [ ]:
# ── App + models ──────────────────────────────────────────────────────────────
app = FastAPI(title="URL Shortener")

In [ ]:
# ── Endpoints ─────────────────────────────────────────────────────────────────
@app.get("/healthz")
def healthz():
    return {"status": "ok"}

In [ ]:
# ── Client ────────────────────────────────────────────────────────────────────
client = TestClient(app, raise_server_exceptions=True)
print(client.get("/healthz").json())

## Helpers

In [ ]:
def call(method: str, path: str, **kwargs):
    r = getattr(client, method)(path, **kwargs)
    body = r.json() if r.content else None
    print(f"{method.upper():6s} {path}  →  {r.status_code}")
    if body is not None:
        print(json.dumps(body, indent=2))
    return r


def df(table: str) -> pd.DataFrame:
    return pd.read_sql(f"SELECT * FROM {table}", conn)


def query(sql: str, *params) -> pd.DataFrame:
    return pd.read_sql(sql, conn, params=list(params) if params else None)


def show_all():
    names = [r[0] for r in conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()]
    for name in names:
        count = conn.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
        print(f"\n── {name} ({count} rows) ──")
        display(pd.read_sql(f"SELECT * FROM {name}", conn))

## Demo

In [ ]:
# Call your endpoints here

## All tables

In [ ]:
show_all()